In [5]:
from helper import load_shots, get_prompt, SUBJECT_FIELDS, TREATMENT_FIELDS

# 1. Load some few-shot examples (e.g., 3 shots in "main" format)
n_shots = 10
raw_examples = load_shots(n_shot=n_shots, cross_split=1, format="raw")
formatted_events = load_shots(n_shot=n_shots, cross_split=1, format="main")

# Pair them up: (text, events)
few_shot_examples = []
for i in range(n_shots):
    few_shot_examples.append((raw_examples[i]["text"], formatted_events[i]))

# 2. Generate a prompt for the "main" task
sentence_to_process = "The patient experienced acute liver failure after starting amiodarone therapy."

main_prompt = get_prompt(
    format_type="main",
    few_shot_examples=few_shot_examples,
    sentence=sentence_to_process,
    order_main=["event_type", "subject", "treatment", "effect"]
)

print("--- MAIN PROMPT ---")
print(main_prompt)


--- MAIN PROMPT ---
For each event, extract the following elements:
- 'event_type': The type of medical event (Adverse_event or Potential_therapeutic_event).
- 'subject': The word or phrase identifying the patient(s) involved in the event.
- 'treatment': The word or phrase describing the medical treatment or medication given to the subject.
- 'effect': The word or phrase describing the medical result or outcome observed.

Recognize all events in the following clinical text and format the output as a list of tuples: [('event_type', 'subject', 'treatment', 'effect'), ...]. If an element is not present, use 'null'.

Text: Successful recovery from interstitial pneumonitis , induced by bicalutamide and leuprorelin acetate given as treatment for prostate cancer .
Events: [('Adverse_event', 'null', 'bicalutamide and leuprorelin acetate', 'interstitial pneumonitis')]
Text: CONCLUSION : The present findings suggest that : ( i ) amantadine probably exerts its anti - dyskinetic effect by acting o

In [6]:
# 3. Generate a prompt for the "sub" task (detailed attributes)
formatted_sub_events = load_shots(n_shot=n_shots, cross_split=1, format="sub")

few_shot_sub_examples = []
for i in range(n_shots):
    few_shot_sub_examples.append((raw_examples[i]["text"], formatted_sub_events[i]))

sub_prompt = get_prompt(
    format_type="sub",
    few_shot_examples=few_shot_sub_examples,
    sentence=sentence_to_process,
    order_main=["event_type", "subject", "treatment", "effect"],
    order_subject=SUBJECT_FIELDS,
    order_treatment=TREATMENT_FIELDS
)

print("\n\n--- SUB PROMPT ---")
print(sub_prompt)




--- SUB PROMPT ---
Extract detailed medical event information from clinical text. Identify the following event types:
- 'Adverse_event': An undesirable medical occurrence after the administration of a pharmaceutical product.
- 'Potential_therapeutic_event': A medical occurrence suggesting potential benefit after the administration of a pharmaceutical product.

For each event, identify the detailed attributes for 'Subject' and 'Treatment':

1. **Subject attributes**:
   - 'subject': The word or phrase identifying the patient(s) involved in the event.
   - 'age': The age of the subject.
   - 'gender': The gender of the subject.
   - 'population': Number of subjects involved.
   - 'race': Race or nationality.
   - 'disorder': Disorders the subject suffers from.

2. **Treatment attributes**:
   - 'treatment': The word or phrase describing the medical treatment or medication given to the subject.
   - 'drug': The specific drug name.
   - 'route': Administration route (e.g., oral, intraven

In [7]:
# Dummy-Vorhersagen (Exakt auf die neue test.json abgestimmt)

# 1. Main-Format Reihenfolge: (event_type, subject, treatment, effect)
dummy_pred_main = [
    [("Adverse_event", "null", "parenteral amiodarone ( 2300 mg in 3 days ) and other measures", "jaundice , marked increase in serum transaminase levels and fall in prothrombin time , and histologic changes of severe centrilobular necrosis were observed in hepatic biopsy")], 
    [("Adverse_event", "a patient , with a 30 - year history of rheumatoid arthritis", "low dose methotrexate weekly over a 10 - month period", "non - Hodgkin lymphoma")],
    [("Adverse_event", "patients", "alum irrigation", "excessive accumulation and toxicity .")],
    [# 4. Multi-Event (Zwei Tuple in einer Liste)
        ("Adverse_event", "a 20 - year - old woman", "oxaprozin", "tense bullae and cutaneous fragility on the face and the back of the hands"),
        ("Adverse_event", "A 44 - year - old man", "naproxen", "tense bullae and cutaneous fragility on the face and the back of the hands")
    ]
]

# 2. Full-Format Reihenfolge in den Tuples:
# Subject: (span, age, gender, population, race, disorder)
# Treatment: (span, drug, route, dosage, time_elapsed, duration, frequency, combination_drug, disorder)
dummy_pred_full = [
    [{"event_type": "Adverse_event", "trigger": "After", "subject": ("null", "null", "null", "null", "null", "null"), "treatment": ("parenteral amiodarone ( 2300 mg in 3 days ) and other measures", "amiodarone", "parenteral", "2300 mg in 3 days", "null", "null", "null", "null", "congestive heart failure"), "effect": "jaundice , marked increase in serum transaminase levels and fall in prothrombin time , and histologic changes of severe centrilobular necrosis were observed in hepatic biopsy"}],
    [{"event_type": "Adverse_event", "trigger": "taking", "subject": ("a patient , with a 30 - year history of rheumatoid arthritis", "null", "null", "a", "null", "rheumatoid arthritis"), "treatment": ("low dose methotrexate weekly over a 10 - month period", "methotrexate", "null", "low dose", "10 - month period", "10 - month period .", "weekly", "null", "rheumatoid arthritis"), "effect": "non - Hodgkin lymphoma"}],
    [{"event_type": "Adverse_event", "trigger": "to", "subject": ("patients", "null", "null", "null", "null", "renal impairment"), "treatment": ("alum irrigation", "alum", "irrigation", "null", "null", "null", "null", "null", "renal impairment"), "effect": "excessive accumulation and toxicity ."}],
    [# 4. Multi-Event (Zwei Dictionaries in einer Liste)
        {"event_type": "Adverse_event", "trigger": "presented", "subject": ("a 20 - year - old woman", "20 - year - old", "woman", "null", "null", "rheumatoid arthritis"), "treatment": ("oxaprozin", "oxaprozin", "null", "null", "null", "null", "null", "null", "rheumatoid arthritis"), "effect": "tense bullae and cutaneous fragility on the face and the back of the hands"},
        {"event_type": "Adverse_event", "trigger": "presented", "subject": ("A 44 - year - old man", "44 - year - old", "man", "null", "null", "chronic low back pain"), "treatment": ("naproxen", "naproxen", "null", "null", "null", "null", "null", "null", "chronic low back pain"), "effect": "tense bullae and cutaneous fragility on the face and the back of the hands"}
    ]
]